# [7.2] Feature Verbalizers - Exercises

Implement the local verbalizer validation loop: collect examples, learn explanation terms from training examples, turn an explanation into predictions, find counterexamples, revise, test intervention direction, and check brevity. The CUDA report uses a real `gelu-1l` residual direction with disjoint train/heldout prompts and a random-direction intervention control, but these exercises are CPU-safe.


In [ ]:
import re
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Literal

import torch as t

chapter = "chapter7_activation_to_language"
section = "part2_feature_verbalizers"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part2_feature_verbalizers.tests as tests

ExampleKind = Literal["top", "bottom", "random", "contrastive"]


@dataclass(frozen=True)
class VerbalizerExample:
    text: str
    score: float
    label: bool
    kind: ExampleKind


@dataclass(frozen=True)
class VerbalizerExampleSet:
    top: tuple[VerbalizerExample, ...]
    bottom: tuple[VerbalizerExample, ...]
    random: tuple[VerbalizerExample, ...]
    contrastive: tuple[VerbalizerExample, ...]


@dataclass(frozen=True)
class ExplanationPredictionReport:
    accuracy: float
    baseline_accuracy: float
    contrastive_accuracy: float
    passes_baseline: bool
    survives_contrastive: bool


@dataclass(frozen=True)
class CounterexampleReport:
    num_counterexamples: int
    counterexamples: tuple[str, ...]


@dataclass(frozen=True)
class InterventionPredictionReport:
    predicted_direction: Literal["increase", "decrease"]
    observed_delta: float
    matches_prediction: bool


@dataclass(frozen=True)
class ExplanationBrevityReport:
    explanation_word_count: int
    examples_word_count: int
    shorter_than_examples: bool


TOKEN_RE = re.compile(r"[a-zA-Z]+")
DEFAULT_STOPWORDS = {"a", "an", "and", "at", "beside", "in", "near", "of", "on", "over", "the", "to"}


## Example Collection

Collect top, bottom, random, and contrastive near-threshold examples. Keep labels and scores attached.


In [ ]:
def gather_verbalizer_examples(
    texts: list[str],
    scores: t.Tensor,
    labels: t.Tensor,
    *,
    k: int = 3,
    threshold: float | None = None,
    seed: int = 0,
) -> VerbalizerExampleSet:
    raise NotImplementedError()


tests.test_gather_verbalizer_examples_covers_top_bottom_random_contrastive(
    gather_verbalizer_examples,
)


## Explanation Predictions

Convert explanation terms into predictions, then score those predictions against labels, a baseline, and a contrastive mask.


In [ ]:
def keyword_explanation_predictions(
    texts: list[str],
    explanation_terms: list[str],
) -> t.Tensor:
    raise NotImplementedError()


def explanation_prediction_report(
    predictions: t.Tensor,
    labels: t.Tensor,
    baseline_predictions: t.Tensor,
    contrastive_mask: t.Tensor,
) -> ExplanationPredictionReport:
    raise NotImplementedError()


tests.test_keyword_predictions_and_explanation_report_use_baseline_and_contrastives(
    keyword_explanation_predictions,
    explanation_prediction_report,
)


## Learn Explanation Terms

A verbalizer should not borrow words from held-out examples. Learn candidate terms from labeled training examples only, then use those terms in the prediction report.


In [ ]:
def learn_verbalizer_terms(
    texts: list[str],
    labels: t.Tensor,
    *,
    top_k: int = 5,
    stopwords: set[str] | None = None,
) -> list[str]:
    raise NotImplementedError()


tests.test_keyword_predictions_do_not_match_substrings(keyword_explanation_predictions)
tests.test_learned_verbalizer_terms_do_not_use_heldout_only_words(learn_verbalizer_terms)


## Counterexamples And Revision

Find failed predictions and revise only when counterexamples actually exist.


In [ ]:
def find_counterexamples(
    texts: list[str],
    predictions: t.Tensor,
    labels: t.Tensor,
    *,
    max_examples: int = 3,
) -> CounterexampleReport:
    raise NotImplementedError()


def revise_explanation(
    explanation: str,
    counterexamples: tuple[str, ...],
    *,
    revision_note: str,
) -> str:
    raise NotImplementedError()


tests.test_counterexamples_and_revision_are_grounded_in_failures(
    find_counterexamples,
    revise_explanation,
)


## Intervention Direction

Check whether intervention scores move in the direction predicted by the explanation.


In [ ]:
def intervention_prediction_report(
    baseline_scores: t.Tensor,
    intervened_scores: t.Tensor,
    *,
    predicted_direction: Literal["increase", "decrease"],
) -> InterventionPredictionReport:
    raise NotImplementedError()


tests.test_intervention_prediction_checks_signed_direction(
    intervention_prediction_report,
)


## Brevity

Compare the explanation length to an examples-only baseline. This is a weak compression check, not a substitute for prediction quality.


In [ ]:
def explanation_brevity_report(
    explanation: str,
    examples: list[str] | tuple[str, ...],
) -> ExplanationBrevityReport:
    raise NotImplementedError()


tests.test_explanation_brevity_compares_against_examples_only_baseline(
    explanation_brevity_report,
)


## Full Verification

After filling in the notebook, compare your implementation to `solutions.py`. The full CUDA path is run separately by `solutions.run_gpu_test(max_vram_gb=24.0)` and should report held-out prediction, contrastive accuracy, zero counterexamples, train/heldout disjointness, learned terms from training examples only, target intervention beating a random-direction control, brevity, and peak VRAM.


## Full Verification Contract

The smoke tests check the local exercise implementation. This final cell checks the committed CUDA verification report for the section-scale run and exposes the same `run_gpu_test` / `run_full_experiment` surface used by the release gate.


In [ ]:
def _load_committed_gpu_report() -> dict:
    import json

    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["within_vram_budget"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


gpu = _load_committed_gpu_report()
{key: gpu[key] for key in [
    "device",
    "preflight_passed",
    "peak_vram_gb",
] if key in gpu}
